# Pipeline — K-Means (Aprendizaje No Supervisado)
### Módulo 3 · Tema 5

---
## ¿Qué es K-Means?
Agrupa los datos en **K clusters** sin usar etiquetas.
La diferencia clave con todo lo anterior: **no existe Y durante el entrenamiento**.

```
Supervisado:    tienes X e Y → el modelo aprende a predecir Y
No supervisado: solo tienes X → el modelo descubre grupos por sí solo
```

## ¿Cómo funciona?
```
1. Elige K puntos aleatorios como centros iniciales (centroides)
2. Asigna cada muestra al centroide más cercano
3. Recalcula los centroides como el promedio de su grupo
4. Repite pasos 2-3 hasta que los grupos no cambien
```

## ¿Cuándo usarlo?
- Segmentar clientes por comportamiento
- Agrupar noticias por tema
- Detectar anomalías
- Comprimir imágenes

## ¿Cómo elegir K?
**Método del codo:** prueba varios K, grafica la inercia (suma de distancias al centroide).
Elige el K donde la curva "dobla" — después de ese punto, añadir más clusters no mejora mucho.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster         import KMeans
from sklearn.cluster         import AgglomerativeClustering  # clustering jerárquico
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import silhouette_score
from sklearn.decomposition   import PCA  # para visualizar en 2D

RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar datos
# ← CAMBIA ESTE BLOQUE por tu dataset
# Nota: aquí NO usamos Y para entrenar — es solo para comparar después
# ═══════════════════════════════════════════════════════════════
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y_real = dataset.target   # guardamos Y solo para comparar al final, NO para entrenar
CLASES = list(dataset.target_names)

print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características")
print("En K-Means NO usamos Y durante el entrenamiento")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Preparar datos
# ═══════════════════════════════════════════════════════════════
# Seleccionar features relevantes (por correlación con Y — usamos Y solo aquí)
df_temp = X.copy()
df_temp['__Y__'] = Y_real
corrs = df_temp.corr()['__Y__'].drop('__Y__')
features_utiles = corrs[corrs.abs() >= 0.5].index.tolist()
X_red = X[features_utiles]

# Escalar — K-Means usa distancias, por eso es muy sensible a la escala
# Sin escalar, una columna con valores grandes (area=1001) domina
# sobre una columna pequeña (smoothness=0.1) aunque no sea más importante
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_red)

print(f"Features seleccionadas: {len(features_utiles)}")
print(f"Datos escalados: {X_sc.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Método del codo para elegir K
# ═══════════════════════════════════════════════════════════════
inercias    = []
silhouettes = []
rango_k     = range(2, 9)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = km.fit_predict(X_sc)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sc, clusters))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfica 1: Método del codo
axes[0].plot(rango_k, inercias, 'bo-', linewidth=2)
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inercia (suma de distancias)')
axes[0].set_title('Método del Codo — elige K donde la curva dobla')
axes[0].grid(True, alpha=0.3)

# Gráfica 2: Silhouette score
# El más alto es el mejor K
axes[1].plot(rango_k, silhouettes, 'ro-', linewidth=2)
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette Score — el pico es el mejor K')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

mejor_k_sil = list(rango_k)[silhouettes.index(max(silhouettes))]
print(f"Mejor K según Silhouette: {mejor_k_sil}")
print("Compara con el gráfico del codo para confirmar")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Entrenar K-Means con K elegido
# ═══════════════════════════════════════════════════════════════
K = mejor_k_sil  # ← o cambia manualmente

# n_init=10: prueba 10 inicializaciones aleatorias y elige la mejor
# Evita quedarse atascado en un mínimo local
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_sc)  # ← NO se le pasa Y

sil = silhouette_score(X_sc, clusters)
print(f"K-Means con K={K}")
print(f"  Silhouette score: {sil:.4f}")
print(f"  (-1 a +1 → más cerca de 1 = clusters mejor separados)")
print()

vals, cnts = np.unique(clusters, return_counts=True)
print("Muestras por cluster:")
for v, c in zip(vals, cnts):
    print(f"  Cluster {v}: {c} muestras ({c/len(clusters)*100:.1f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Visualizar clusters en 2D con PCA
# PCA reduce muchas dimensiones a 2 para poder graficar
# ═══════════════════════════════════════════════════════════════
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores = ['#3B8BD4', '#D85A30', '#2ca02c', '#9467bd', '#8c564b']

# Gráfica 1: clusters encontrados por K-Means
for c in range(K):
    mascara = clusters == c
    axes[0].scatter(X_2d[mascara, 0], X_2d[mascara, 1],
                    c=colores[c], s=20, alpha=0.6, label=f'Cluster {c}')
axes[0].set_title(f'Clusters K-Means (K={K})')
axes[0].set_xlabel('Componente 1 (PCA)')
axes[0].set_ylabel('Componente 2 (PCA)')
axes[0].legend()

# Gráfica 2: clases reales (para comparar)
for c in range(len(CLASES)):
    mascara = Y_real == c
    axes[1].scatter(X_2d[mascara, 0], X_2d[mascara, 1],
                    c=colores[c], s=20, alpha=0.6, label=CLASES[c])
axes[1].set_title('Clases reales (Y)')
axes[1].set_xlabel('Componente 1 (PCA)')
axes[1].legend()

plt.suptitle('Comparación: clusters encontrados vs clases reales', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Varianza explicada por PCA: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print("Si los clusters se parecen a las clases reales → K-Means los encontró bien")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 7 — Clustering Jerárquico (alternativa a K-Means)
# ═══════════════════════════════════════════════════════════════
# El clustering jerárquico no necesita que definas K de antemano
# Construye un árbol de agrupaciones (dendrograma)

hier = AgglomerativeClustering(n_clusters=K, linkage='ward')
clusters_hier = hier.fit_predict(X_sc)

sil_hier = silhouette_score(X_sc, clusters_hier)
print("Clustering Jerárquico vs K-Means:")
print(f"  K-Means    — Silhouette: {sil:.4f}")
print(f"  Jerárquico — Silhouette: {sil_hier:.4f}")
print()
print("El que tenga mayor Silhouette agrupa mejor los datos.")